In [ ]:
# %% [markdown]
# # Anveshan DataSprint 2.0 — Hindi/Hinglish Intent NLU (56 classes, Macro-F1)
# Pipeline (final inference model <= 300M params):
#   Stage A  TF-IDF (romanized + phonetic + consonant-skeleton views) + LinearSVC  -> fast, CPU
#   Stage B  MuRIL-base (237M) fine-tune, noise/script-mix augmentation, K-fold OOF + full-data model
#   Stage C  Blend (weight tuned on OOF) + per-class bias tuning for Macro-F1 + Class-55 fallback threshold
# A submission file is written after Stage A and again after Stage C, so you always have something to submit.

# %% CONFIG
import os, re, glob, json, time, math, random, unicodedata, gc
import numpy as np, pandas as pd

CFG = dict(
    DATA_DIR       = os.environ.get("DATA_DIR", ""),        # "" = auto-detect under /kaggle/input
    OUT_DIR        = os.environ.get("OUT_DIR", "/kaggle/working"),
    MODEL_NAME     = os.environ.get("MODEL_NAME", "google/muril-base-cased"),  # alt: "xlm-roberta-base" (278M)
    FAST_MODE      = True,     # True: CV on fold 0 only (quick first submission). False: all folds (better tuning)
    N_FOLDS        = 5,
    RUN_TRANSFORMER= True,
    TRAIN_FULL     = True,     # train one model on 100% data for test preds (keeps final pipeline <= 300M params)
    EPOCHS         = 6,
    BATCH_SIZE     = 32,
    LR             = 4e-5,
    WEIGHT_DECAY   = 0.01,
    WARMUP         = 0.1,
    MAX_LEN        = 64,       # auto-shrunk to 99th pct token length
    LABEL_SMOOTH   = 0.1,
    USE_FGM        = True,     # adversarial training on embeddings: usually +0.5-1 F1, ~1.8x train time
    FGM_EPS        = 1.0,
    SVC_CS         = (0.3, 0.7, 1.5),
    USE_LOOKUP     = True,     # exact-duplicate memory: test text seen in train with one unambiguous label
    CLASS_WEIGHT_POW = 0.5,    # 0 = none, 0.5 = sqrt inverse-frequency (helps tail classes for Macro-F1)
    AUG_PROB       = 0.35,     # prob of ASR-style noise per training sample
    TRANSLIT_PROB  = 0.25,     # prob of converting Devanagari -> Roman (script-mix robustness)
    OOS_LABEL      = None,     # None = auto (55 if numeric labels, else name containing oos/out/fallback)
    SEED           = 42,
)
os.makedirs(CFG["OUT_DIR"], exist_ok=True)

def seed_all(s):
    random.seed(s); np.random.seed(s); os.environ["PYTHONHASHSEED"] = str(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError: pass
seed_all(CFG["SEED"])

# %% LOAD DATA (auto-detects file + column names)
def find_data_dir():
    if CFG["DATA_DIR"]: return CFG["DATA_DIR"]
    hits = glob.glob("/kaggle/input/**/train*.csv", recursive=True)
    if not hits:
        hits = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    if not hits:
        raise FileNotFoundError("No .csv found under /kaggle/input — attach the competition data.")
    return os.path.dirname(hits[0])

DATA_DIR = find_data_dir()

def pick(patterns):
    if isinstance(patterns, str):
        patterns = [patterns]
    for pat in patterns:
        hits = sorted(glob.glob(os.path.join(DATA_DIR, "**", pat), recursive=True))
        if hits:
            return hits[0]
    return None

all_csvs = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
print("csv files found:", [os.path.relpath(p, DATA_DIR) for p in all_csvs])

train_path = pick(["train*.csv", "*train*.csv"])
if not train_path:
    raise FileNotFoundError(f"Could not find train CSV under {DATA_DIR}. Found: {all_csvs}")

test_path = pick(["test*.csv", "*test*.csv", "eval*.csv", "*eval*.csv", "val*.csv", "*val*.csv"])
sub_path = pick(["sample*sub*.csv", "*sample*sub*.csv", "*sub*.csv"])

train = pd.read_csv(train_path)
if test_path:
    test = pd.read_csv(test_path)
else:
    print("WARNING: No test CSV found; using a small slice of train as dummy test so the pipeline still runs.")
    test = train.iloc[:min(50, len(train))].copy()

sample = pd.read_csv(sub_path) if sub_path else None
print("train", train.shape, list(train.columns))
print("test", test.shape, list(test.columns))

ID_COL = sample.columns[0] if sample is not None else next((c for c in test.columns if "id" in c.lower()), test.columns[0])
TARGET_COL = sample.columns[1] if (sample is not None and len(sample.columns) > 1) else None
lab_cands = [c for c in train.columns if c not in test.columns and c != ID_COL]
LABEL_COL = TARGET_COL if TARGET_COL in train.columns else (lab_cands[0] if lab_cands else train.columns[-1])
obj_cols = [c for c in test.columns if not pd.api.types.is_numeric_dtype(test[c]) and c != ID_COL]
TEXT_COL = max(obj_cols, key=lambda c: test[c].astype(str).str.len().mean()) if obj_cols else [c for c in train.columns if c not in (ID_COL, LABEL_COL)][0]
TARGET_COL = TARGET_COL or LABEL_COL
print(f"ID={ID_COL}  TEXT={TEXT_COL}  LABEL={LABEL_COL}  SUB_TARGET={TARGET_COL}")

train = train.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
test[TEXT_COL] = test[TEXT_COL].fillna("")
classes = np.array(sorted(train[LABEL_COL].unique()))
cls2idx = {c: i for i, c in enumerate(classes)}
y = train[LABEL_COL].map(cls2idx).values
K = len(classes)
if CFG["OOS_LABEL"] is not None: OOS = cls2idx[CFG["OOS_LABEL"]]
elif np.issubdtype(classes.dtype, np.number) and 55 in cls2idx: OOS = cls2idx[55]
else:
    m = [i for i, c in enumerate(classes) if re.search(r"oos|out|fallback|scope|other", str(c).lower())]
    OOS = m[0] if m else K - 1
print(f"{K} classes, OOS idx={OOS} ({classes[OOS]})")
print(train[LABEL_COL].value_counts().describe())

# %% TEXT NORMALIZATION + DEVANAGARI -> ROMAN TRANSLITERATION (no external deps)
V_IND = dict(zip("अआइईउऊऋएऐओऔऑ", ["a","aa","i","ee","u","oo","ri","e","ai","o","au","o"]))
MATRA = dict(zip("ािीुूृेैोौॉॅ", ["aa","i","ee","u","oo","ri","e","ai","o","au","o","e"]))
CONS = dict(zip("कखगघङचछजझञटठडढणतथदधनपफबभमयरलवशषसह",
    "k kh g gh n ch chh j jh n t th d dh n t th d dh n p ph b bh m y r l v sh sh s h".split()))
CONS.update({"क़":"q","ख़":"kh","ग़":"g","ज़":"z","ड़":"d","ढ़":"rh","फ़":"f","य़":"y"})
NUKTA_FIX = {"क":"क़","ख":"ख़","ग":"ग़","ज":"ज़","ड":"ड़","ढ":"ढ़","फ":"फ़"}
DIGITS = dict(zip("०१२३४५६७८९", "0123456789"))
DEV_RE = re.compile(r"[ऀ-ॿ]")

def translit_word(w):
    out, i, n = [], 0, len(w)
    while i < n:
        ch = w[i]
        if i + 1 < n and w[i+1] == "़" and ch in NUKTA_FIX: ch = NUKTA_FIX[ch]; i += 1
        if ch in CONS:
            out.append(CONS[ch]); nxt = w[i+1] if i + 1 < n else ""
            if nxt in MATRA: out.append(MATRA[nxt]); i += 1
            elif nxt == "्": i += 1           # virama: no inherent vowel
            else: out.append("a*")                 # inherent schwa (may be deleted)
        elif ch in V_IND: out.append(V_IND[ch])
        elif ch in MATRA: out.append(MATRA[ch])
        elif ch in "ंँ": out.append("n")
        elif ch == "ः": out.append("h")
        elif ch in DIGITS: out.append(DIGITS[ch])
        elif ch in "़्": pass
        else: out.append(ch)
        i += 1
    if len(out) > 2 and out[-1] == "a*": out.pop()  # Hindi word-final schwa deletion
    return "".join(out).replace("a*", "a")

def to_roman(s):
    if not DEV_RE.search(s): return s
    return re.sub(r"[ऀ-ॿ]+", lambda m: translit_word(m.group()), s)

def clean_basic(s):
    s = unicodedata.normalize("NFKC", str(s))
    s = re.sub(r"https?://\S+", " url ", s)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)          # "bhaiii" -> "bhaii"
    s = s.replace("।", " ")
    return re.sub(r"\s+", " ", s).strip()

def norm_roman(s):
    s = to_roman(clean_basic(s)).lower()
    s = re.sub(r"\d", "0", s)
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

SQUASH = [("aa","a"),("ee","i"),("ii","i"),("oo","u"),("uu","u"),("ph","f"),("w","v"),("z","j"),
          ("q","k"),("ck","k"),("x","ks"),("y","i")]
def squash(s):
    for a, b in SQUASH: s = s.replace(a, b)
    s = re.sub(r"([kgcjtdpbs])h", r"\1", s)        # kh->k, th->t ...
    return re.sub(r"(.)\1+", r"\1", s)

def skeleton(s):  # consonant skeleton: robust to dropped matras / vowel typos
    return " ".join(w[0] + re.sub(r"[aeiou]", "", w[1:]) if w else w for w in s.split())

for df in (train, test):
    df["clean"] = df[TEXT_COL].map(clean_basic)      # transformer input (keeps script)
    df["roman"] = df[TEXT_COL].map(norm_roman)
    df["squash"] = df["roman"].map(squash)
    df["skel"] = df["squash"].map(skeleton)
    df["orig"] = df["clean"].str.lower().map(lambda s: re.sub(r"\s+", " ", re.sub(r"[^\w\s\u0900-\u097F]", " ", s)).strip())
print(train[[TEXT_COL, "roman", "squash", "skel"]].head(5).to_string())

# %% FOLDS + METRIC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
# Group by normalized text: duplicate utterances never sit on both sides of a split,
# so OOF scores (and every threshold tuned on them) are honest, not inflated by memorized duplicates.
groups = pd.factorize(train["roman"])[0]
print(f"unique normalized texts: {groups.max()+1} / {len(train)} rows")
sgkf = StratifiedGroupKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED"])
folds = np.zeros(len(train), int)
for f, (_, va) in enumerate(sgkf.split(train, y, groups)): folds[va] = f
CV_FOLDS = [0] if CFG["FAST_MODE"] else list(range(CFG["N_FOLDS"]))

def macro_f1(yt, yp):   # fast numpy version (same as sklearn average="macro" over all K labels)
    tp = np.bincount(yt[yt == yp], minlength=K).astype(float)
    pc = np.bincount(yp, minlength=K); tc = np.bincount(yt, minlength=K)
    d = pc + tc; return float(np.mean(np.where(d > 0, 2 * tp / np.maximum(d, 1), 0.0)))
def softmax(z, T=1.0):
    z = z * T; z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

# %% STAGE A — TF-IDF multi-view + LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from scipy.sparse import hstack

t0 = time.time()
VIEWS = [("roman", dict(analyzer="char_wb", ngram_range=(1, 5), max_features=200_000)),
         ("roman", dict(analyzer="word", ngram_range=(1, 2), max_features=100_000, token_pattern=r"(?u)\b\w+\b")),
         ("squash", dict(analyzer="char_wb", ngram_range=(2, 5), max_features=150_000)),
         ("skel", dict(analyzer="char_wb", ngram_range=(2, 4), max_features=80_000)),
         ("orig", dict(analyzer="char_wb", ngram_range=(1, 4), max_features=150_000))]   # original script
Xtr_parts, Xte_parts, VECS = [], [], []
for col, kw in VIEWS:
    vec = TfidfVectorizer(sublinear_tf=True, min_df=2, dtype=np.float32, **kw)
    vec.fit(pd.concat([train[col], test[col]]))       # unsupervised IDF on train+test
    VECS.append(vec); Xtr_parts.append(vec.transform(train[col])); Xte_parts.append(vec.transform(test[col]))
Xtr = hstack(Xtr_parts).tocsr(); Xte = hstack(Xte_parts).tocsr()
print("tfidf dims:", Xtr.shape, f"{time.time()-t0:.1f}s")

def full_scores(clf, X):   # align to K columns even if a class is missing from a fold's fit
    s = clf.decision_function(X); out = np.full((X.shape[0], K), -5.0, np.float32)
    out[:, clf.classes_] = s; return out

best_svc = (-1, None, None)
for C in CFG["SVC_CS"]:                               # SVC is cheap: full CV for every C
    oof = np.zeros((len(train), K), np.float32)
    for f in range(CFG["N_FOLDS"]):
        tr, va = folds != f, folds == f
        oof[va] = full_scores(LinearSVC(C=C, class_weight="balanced", max_iter=5000).fit(Xtr[tr], y[tr]), Xtr[va])
    s = macro_f1(y, oof.argmax(1)); print(f"SVC C={C}: OOF macro-F1 {s:.4f}  ({time.time()-t0:.0f}s)")
    if s > best_svc[0]: best_svc = (s, C, oof)
_, C_SVC, svc_oof = best_svc
svc_full = LinearSVC(C=C_SVC, class_weight="balanced", max_iter=5000).fit(Xtr, y)
svc_test = full_scores(svc_full, Xte)
print(f"SVC best C={C_SVC}  OOF macro-F1: {best_svc[0]:.4f}")

# temperature so SVC scores become calibrated-ish probabilities (for blending)
def nll(P, yy): return -np.log(P[np.arange(len(yy)), yy] + 1e-9).mean()
T_SVC = min([0.5, 1, 2, 3, 5, 8, 12, 20], key=lambda T: nll(softmax(svc_oof, T), y))
print("SVC temperature:", T_SVC)

def write_sub(pred_idx, name):
    out = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: classes[pred_idx]})
    if sample is not None: out = sample[[ID_COL]].merge(out, on=ID_COL, how="left")
    p = os.path.join(CFG["OUT_DIR"], name); out.to_csv(p, index=False); print("wrote", p, out.shape)
    return out

write_sub(svc_test.argmax(1), "submission_tfidf.csv")
write_sub(svc_test.argmax(1), "submission.csv")     # safety: overwritten after Stage C

# %% STAGE B — Transformer fine-tune (MuRIL)
tf_oof = np.full((len(train), K), np.nan, np.float32)
tf_test = None
if CFG["RUN_TRANSFORMER"]:
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = dev.type == "cuda"
    print("device:", dev, torch.cuda.get_device_name(0) if use_amp else "")
    tok = AutoTokenizer.from_pretrained(CFG["MODEL_NAME"])
    lens = [len(tok(t)["input_ids"]) for t in train["clean"].sample(min(3000, len(train)), random_state=0)]
    MAX_LEN = int(min(CFG["MAX_LEN"], max(16, np.percentile(lens, 99) + 4)))
    print("MAX_LEN:", MAX_LEN)

    VOW = "aeiou"; MATRAS = "ािीुूेैोौंँ्"
    def noisy(s):
        """ASR / typing noise: drop matra/vowel, swap, duplicate, split or merge words."""
        ws = s.split()
        if not ws: return s
        for _ in range(random.randint(1, 2)):
            i = random.randrange(len(ws)); w = ws[i]; op = random.random()
            if len(w) < 3: continue
            j = random.randrange(1, len(w))
            if op < 0.35:                                    # drop a vowel / matra
                idx = [k for k, c in enumerate(w) if c in VOW or c in MATRAS]
                if idx: k = random.choice(idx); w = w[:k] + w[k+1:]
            elif op < 0.55: w = w[:j-1] + w[j] + w[j-1] + w[j+1:]   # swap
            elif op < 0.7:  w = w[:j] + w[j-1] + w[j:]               # duplicate char
            elif op < 0.85: w = w[:j] + " " + w[j:]                  # split word
            elif i + 1 < len(ws): w = w + ws.pop(i + 1)             # merge words
            ws[i] = w
        return " ".join(ws)

    def augment(s):
        if DEV_RE.search(s) and random.random() < CFG["TRANSLIT_PROB"]: s = to_roman(s).lower()
        if random.random() < CFG["AUG_PROB"]: s = noisy(s)
        return s

    def make_loader(texts, labels=None, train_mode=False, bs=None):
        idx = list(range(len(texts)))
        def collate(b):
            tx = [augment(texts[i]) if train_mode else texts[i] for i in b]
            enc = tok(tx, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
            if labels is not None: enc["labels"] = torch.tensor([labels[i] for i in b])
            return enc
        return DataLoader(idx, batch_size=bs or (CFG["BATCH_SIZE"] if train_mode else 128),
                          shuffle=train_mode, collate_fn=collate, num_workers=2, drop_last=False)

    counts = np.bincount(y, minlength=K).astype(np.float32)
    cw = (counts.max() / np.maximum(counts, 1)) ** CFG["CLASS_WEIGHT_POW"]
    cw = torch.tensor(cw / cw.mean(), dtype=torch.float32, device=dev)

    @torch.no_grad()
    def predict(model, texts):
        model.eval(); outs = []
        for b in make_loader(texts):
            b = {k: v.to(dev) for k, v in b.items()}
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                outs.append(model(**b).logits.float().cpu())
        return torch.cat(outs).numpy()

    def train_model(tr_texts, tr_y, va_texts=None, va_y=None, epochs=None, tag=""):
        seed_all(CFG["SEED"])
        model = AutoModelForSequenceClassification.from_pretrained(CFG["MODEL_NAME"], num_labels=K, ignore_mismatched_sizes=True).to(dev)
        epochs = epochs or CFG["EPOCHS"]
        dl = make_loader(tr_texts, tr_y, train_mode=True)
        nd = ["bias", "LayerNorm.weight", "layer_norm.weight"]
        params = [{"params": [p for n, p in model.named_parameters() if not any(x in n for x in nd)], "weight_decay": CFG["WEIGHT_DECAY"]},
                  {"params": [p for n, p in model.named_parameters() if any(x in n for x in nd)], "weight_decay": 0.0}]
        opt = torch.optim.AdamW(params, lr=CFG["LR"])
        steps = epochs * len(dl)
        sch = get_linear_schedule_with_warmup(opt, int(CFG["WARMUP"] * steps), steps)
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
        loss_fn = nn.CrossEntropyLoss(weight=cw, label_smoothing=CFG["LABEL_SMOOTH"])
        best, best_state, best_ep, best_va = -1, None, epochs, None
        for ep in range(epochs):
            model.train(); t0 = time.time(); tot = 0
            for b in dl:
                b = {k: v.to(dev) for k, v in b.items()}; lab = b.pop("labels")
                with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                    loss = loss_fn(model(**b).logits.float(), lab)
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
                if CFG["USE_FGM"]:                       # FGM: perturb word embeddings along the gradient
                    emb = model.get_input_embeddings().weight; g = emb.grad
                    gn = torch.norm(g) if g is not None else None
                    if gn is not None and torch.isfinite(gn) and gn > 0:
                        delta = CFG["FGM_EPS"] * g / gn; emb.data.add_(delta)
                        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                            adv = loss_fn(model(**b).logits.float(), lab)
                        scaler.scale(adv).backward(); emb.data.sub_(delta)
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sch.step(); tot += loss.item()
            msg = f"[{tag}] ep{ep+1}/{epochs} loss={tot/len(dl):.4f} {time.time()-t0:.0f}s"
            if va_texts is not None:
                lv = predict(model, va_texts); s = macro_f1(va_y, lv.argmax(1)); msg += f"  val macroF1={s:.4f}"
                if s > best:
                    best, best_ep, best_va = s, ep + 1, lv
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(msg, flush=True)
        if best_state is not None: model.load_state_dict(best_state)
        return model, best_ep, best_va

    best_eps = []
    tr_text = train["clean"].tolist()
    for f in CV_FOLDS:
        tr, va = np.where(folds != f)[0], np.where(folds == f)[0]
        model, bep, lv = train_model([tr_text[i] for i in tr], y[tr], [tr_text[i] for i in va], y[va], tag=f"fold{f}")
        tf_oof[va] = lv; best_eps.append(bep)
        if not CFG["TRAIN_FULL"]:
            lt = predict(model, test["clean"].tolist()); tf_test = lt if tf_test is None else tf_test + lt
        if f == CV_FOLDS[0] and not CFG["TRAIN_FULL"]:
            model.save_pretrained(f"{CFG['OUT_DIR']}/model"); tok.save_pretrained(f"{CFG['OUT_DIR']}/model")
        del model; gc.collect(); torch.cuda.empty_cache()
    np.save(f"{CFG['OUT_DIR']}/tf_oof.npy", tf_oof)

    if CFG["TRAIN_FULL"]:
        full_ep = int(round(np.median(best_eps)))
        print("training full-data model for", full_ep, "epochs")
        model, _, _ = train_model(tr_text, y, epochs=full_ep, tag="full")
        tf_test = predict(model, test["clean"].tolist())
        model.save_pretrained(f"{CFG['OUT_DIR']}/model"); tok.save_pretrained(f"{CFG['OUT_DIR']}/model")
    np.save(f"{CFG['OUT_DIR']}/tf_test.npy", tf_test)

# %% STAGE C — Blend + Macro-F1 post-processing (bias per class + OOS threshold)
mask = ~np.isnan(tf_oof[:, 0]) if CFG["RUN_TRANSFORMER"] else np.ones(len(train), bool)
yv = y[mask]
Ps_oof, Ps_test = softmax(svc_oof[mask], T_SVC), softmax(svc_test, T_SVC)
if CFG["RUN_TRANSFORMER"]:
    Pt_oof, Pt_test = softmax(tf_oof[mask]), softmax(tf_test)
    print(f"OOF on {mask.sum()} rows | SVC {macro_f1(yv, Ps_oof.argmax(1)):.4f} | TF {macro_f1(yv, Pt_oof.argmax(1)):.4f}")
    W = max(np.arange(0, 1.01, 0.05), key=lambda w: macro_f1(yv, (w*Pt_oof + (1-w)*Ps_oof).argmax(1)))
    P_oof, P_test = W*Pt_oof + (1-W)*Ps_oof, W*Pt_test + (1-W)*Ps_test
    print(f"blend weight(transformer)={W:.2f}  blend F1={macro_f1(yv, P_oof.argmax(1)):.4f}")
else:
    W = 0.0; P_oof, P_test = Ps_oof, Ps_test

L_oof, L_test = np.log(P_oof + 1e-9), np.log(P_test + 1e-9)
bias = np.zeros(K, np.float32)
base = macro_f1(yv, L_oof.argmax(1))
if len(yv) >= 1500:     # per-class bias only when enough OOF rows to avoid overfitting
    grid = np.arange(-1.5, 1.51, 0.25)
    for _ in range(2):
        for c in range(K):
            best_g = bias[c]; best_s = macro_f1(yv, (L_oof + bias).argmax(1))
            for g in grid:
                bias[c] = g; s = macro_f1(yv, (L_oof + bias).argmax(1))
                if s > best_s + 1e-4: best_s, best_g = s, g
            bias[c] = best_g
    bias *= 0.7          # shrink toward 0 for generalization
print(f"bias tuning: {base:.4f} -> {macro_f1(yv, (L_oof + bias).argmax(1)):.4f}")

def decide(L, tau):
    p = (L + bias).argmax(1); conf = np.exp(L).max(1)
    p = p.copy(); p[conf < tau] = OOS; return p
TAU = max(np.arange(0, 0.61, 0.025), key=lambda t: macro_f1(yv, decide(L_oof, t)))
final_oof = macro_f1(yv, decide(L_oof, TAU))
print(f"OOS threshold tau={TAU:.3f}  FINAL OOF macro-F1={final_oof:.4f}")
oos_f1 = f1_score(yv == OOS, decide(L_oof, TAU) == OOS); print(f"class-55 (OOS) F1={oos_f1:.4f}")

pred = decide(L_test, TAU)
if CFG["USE_LOOKUP"]:   # exact normalized duplicate of a train row with a single consistent label
    lk = train.groupby("roman")[LABEL_COL].agg(lambda s: s.map(cls2idx).iloc[0] if s.nunique() == 1 else -1)
    hit = test["roman"].map(lk).fillna(-1).astype(int).values
    m = hit >= 0; changed = int((pred[m] != hit[m]).sum())
    print(f"lookup: {m.sum()} test rows are exact train duplicates, {changed} predictions corrected")
    pred = np.where(m, hit, pred)
write_sub(pred, "submission.csv")
json.dump(dict(W=float(W), C_SVC=float(C_SVC), T_SVC=float(T_SVC), TAU=float(TAU),
               bias=bias.tolist(), oof_f1=float(final_oof)), open(f"{CFG['OUT_DIR']}/postproc.json", "w"))
print(pd.Series(classes[pred]).value_counts().head(10))

# %% LATENCY CHECK — full pipeline, single query (what you demo live; target < 50 ms)
import joblib
joblib.dump(dict(vecs=VECS, views=VIEWS, svc=svc_full, T=T_SVC, W=W, bias=bias, tau=TAU, classes=classes,
                 oos=OOS), f"{CFG['OUT_DIR']}/tfidf_pipeline.joblib")
def views_of(text):
    r = norm_roman(text); sq = squash(r); c = clean_basic(text)
    o = re.sub(r"\s+", " ", re.sub(r"[^\w\s\u0900-\u097F]", " ", c.lower())).strip()
    return {"roman": r, "squash": sq, "skel": skeleton(sq), "orig": o, "clean": c}

def predict_one(text):
    v = views_of(text)
    x = hstack([vec.transform([v[col]]) for (col, _), vec in zip(VIEWS, VECS)]).tocsr()
    p = softmax(full_scores(svc_full, x), T_SVC)
    if CFG["RUN_TRANSFORMER"] and W > 0:
        enc = tok([v["clean"]], truncation=True, max_length=MAX_LEN, return_tensors="pt").to(dev)
        with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
            p = W * softmax(model(**enc).logits.float().cpu().numpy()) + (1 - W) * p
    L = np.log(p + 1e-9) + bias
    return OOS if p.max() < TAU else int(L.argmax())

if CFG["RUN_TRANSFORMER"]: import torch; model.eval()
for q in ["bhai kal subah 7 baje ka alarm laga de", "मेरी शॉपिंग लिस्ट में दूध जोड़ो", "bitcoin ka price kya hai"]:
    predict_one(q); t = time.perf_counter()
    for _ in range(20): c = predict_one(q)
    print(f"{(time.perf_counter()-t)/20*1000:.1f} ms  -> {classes[c]}  | {q}")
n_params = svc_full.coef_.size + (sum(p.numel() for p in model.parameters()) if CFG["RUN_TRANSFORMER"] else 0)
print(f"pipeline params: {n_params/1e6:.1f}M (limit 300M)")